In [38]:
!pip install pycldf

In [46]:
import pandas as pd
from pycldf import Wordlist

extant = pd.read_csv(
    "merged_final_with_geo_and_sources.tsv",
    sep="\t"
)

proto = pd.read_csv(
    "protoforms_final_with_sources.tsv",
    sep="\t"
)

# Protoforms have no coordinates
proto["LATITUDE"] = pd.NA
proto["LONGITUDE"] = pd.NA

print(extant.columns.tolist())
print(proto.columns.tolist())

# Merge datasets
df = pd.concat(
    [extant, proto],
    ignore_index=True
)

# Clean missing values
df = df.replace("nan", pd.NA)
df["FORM"] = df["FORM"].replace("", pd.NA)

# Remove empty forms
df = df[df["FORM"].notna()].copy()

# =========================
# LANGUAGES TABLE
# =========================

languages = (
    df[
        [
            "DOCULECT",
            "LANGUAGE_NAME",
            "FAMILY",
            "LATITUDE",
            "LONGITUDE"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
    .copy()
)

languages["ID"] = [
    f"l{i}"
    for i in range(1, len(languages) + 1)
]

languages = languages.rename(columns={
    "DOCULECT": "Glottocode",
    "LANGUAGE_NAME": "Name",
    "FAMILY": "Family",
    "LATITUDE": "Latitude",
    "LONGITUDE": "Longitude"
})

# Mapping glottocode -> language ID
language_map = dict(
    zip(languages["Glottocode"], languages["ID"])
)

# =========================
# PARAMETERS TABLE
# =========================

parameters = (
    df[
        ["CONCEPT", "CONCEPTICON_ID"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
    .copy()
)

parameters["ID"] = [
    f"p{i}"
    for i in range(1, len(parameters) + 1)
]

# Create mapping BEFORE renaming
parameter_map = dict(
    zip(
        parameters["ID"],
        parameters["CONCEPTICON_ID"]
    )
)

parameters = parameters[
    [
        "ID",
        "CONCEPT",
        "CONCEPTICON_ID"
    ]
]

# =========================
# FORMS TABLE
# =========================

form_df = df.copy()

form_df["Language_ID"] = (
    form_df["DOCULECT"]
    .map(language_map)
)

form_df["Parameter_ID"] = (
    form_df["CONCEPTICON_ID"]
    .map(parameter_map)
)

form_df["ID"] = [
    f"f{i}"
    for i in range(1, len(form_df) + 1)
]

forms = form_df.rename(columns={
    "FORM": "Form",
    "TOKENS": "Segments"
})

forms = forms[
    [
        "ID",
        "Language_ID",
        "Parameter_ID",
        "Form",
        "Segments",
        "Source"
    ]
]

# =========================
# EXPORT CSVS
# =========================

languages.to_csv("languages.csv", index=False)
parameters.to_csv("parameters.csv", index=False)
forms.to_csv("forms.csv", index=False)

['ORIGINAL_GLOSS', 'CONCEPT', 'CONCEPTICON_ID', 'LANGUAGE_NAME', 'DOCULECT', 'FAMILY', 'FORM', 'TOKENS', 'LATITUDE', 'LONGITUDE', 'Source']
['ORIGINAL_GLOSS', 'CONCEPT', 'CONCEPTICON_ID', 'LANGUAGE_NAME', 'DOCULECT', 'FAMILY', 'FORM', 'TOKENS', 'Source', 'LATITUDE', 'LONGITUDE']


/tmp/ipykernel_10426/4161346883.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(


In [47]:
ds = Wordlist.in_dir(".")

# Only add missing tables
ds.add_component("LanguageTable")
ds.add_component("ParameterTable")

# Add Family column
ds.add_columns(
    "LanguageTable",
    {
        "name": "Family",
        "datatype": "string"
    }
)

ds.add_columns(
    "parameters.csv",
    {
        "name": "CONCEPT",
        "datatype": "string"
    },
    {
        "name": "CONCEPTICON_ID",
        "datatype": "string"
    }
)

# Set separators
ds["FormTable", "Segments"].separator = " "
ds["FormTable", "Source"].separator = ";"

# Write dataset
ds.write(
    FormTable=forms.to_dict("records"),
    LanguageTable=languages.to_dict("records"),
    ParameterTable=parameters.to_dict("records")
)

PosixPath('Wordlist-metadata.json')